# 04 - Feature Engineering

**Project:** Student Placement & Salary Prediction  
**Goal of this notebook:** salary model ke liye students aur features select karna, feature importance se check karna, aur train/test split banana.

**Notebook order:** 01 Data Understanding -> 02 EDA -> 03 Data Preprocessing -> 04 Feature Engineering -> 05 Model Training -> 06 Model Evaluation. Ye notebook `notebook/` folder ke andar se run karo (paths `../datasets` aur `../models` relative hain).

## 1. Libraries aur encoded data load karna
Notebook 03 ka output (`encoded_data.csv`) aur encoders load karte hain.

In [ ]:
import json, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../datasets/processed/encoded_data.csv')
encoders = pickle.load(open('../models/encoders.pkl', 'rb'))
TARGET = 'salary_package_lpa'
print(df.shape)

## 2. Salary model ke liye sirf Placed students
Notebook 01 me dekha ki Not Placed ka salary 0 hai. Salary predict karne ke liye sirf Placed students use karte hain. `Placed` ka encoded number encoder se nikalte hain (hard-code nahi karte).

In [ ]:
placed_value = encoders['placement_status'].transform(['Placed'])[0]
salary_df = df[df['placement_status'] == placed_value].copy()
print('Placed code =', placed_value, '| rows for salary model =', len(salary_df))

## 3. Kaun se features candidates hain?
Identifier (`student_id`) aur targets (`placement_status`, `salary_package_lpa`) ko hata ke baaki saare columns candidate features hain.

In [ ]:
candidates = [c for c in salary_df.columns if c not in ['student_id', 'placement_status', TARGET]]
print(len(candidates), 'candidate features')
candidates

## 4. Feature importance nikalna
Ek chhota Random Forest (100 trees, depth 12) sab candidate features par train karke dekhte hain ki kaun se features salary me kaam aate hain. Saath me correlation bhi dekhte hain. (Ye sirf feature selection ke liye hai; final models notebook 05 me train hote hain.)

In [ ]:
fs = salary_df.sample(20000, random_state=42)
rf_fs = RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_fs.fit(fs[candidates], fs[TARGET])
imp = pd.DataFrame({
    'feature': candidates,
    'rf_importance': rf_fs.feature_importances_,
    'correlation_with_salary': salary_df[candidates].corrwith(salary_df[TARGET]).values
}).sort_values('rf_importance', ascending=False).reset_index(drop=True)
imp.round(4)

## 5. Importance ka chart

In [ ]:
top = imp.sort_values('rf_importance')
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top['feature'], top['rf_importance'])
ax.set_title('Random Forest feature importance (salary)')
plt.tight_layout()
plt.show()

## 6. Final features chunna
Project me **12 features** use hue hain (dashboard ke Salary Prediction form me bhi yahi 12 inputs hain). Ye list dashboard ke `utils.py` ke `SELECTED_FEATURES` jaisi hi hai.

**Honest note:** upar ke importance table se dikhta hai ki sirf kuch features (`cgpa`, `coding_skill_score`, `internships_count`) salary me asli kaam karte hain (teeno milke ~81% importance); baaki selected features ki importance bahut kam hai. Dashboard same rehna tha isliye 12 ki list nahi badli.

In [ ]:
SELECTED_FEATURES = [
    'cgpa', 'branch', 'college_tier', 'internships_count', 'projects_count',
    'certifications_count', 'coding_skill_score', 'aptitude_score',
    'communication_skill_score', 'mock_interview_score',
    'attendance_percentage', 'backlogs'
]
imp['selected'] = imp['feature'].isin(SELECTED_FEATURES)
print('Selected:', len(SELECTED_FEATURES))
print('Selected features ki importance ka total:', round(imp.loc[imp['selected'], 'rf_importance'].sum(), 4))
imp[['feature', 'rf_importance', 'selected']].round(4)

## 7. X aur y banana, phir Train/Test split
80% training, 20% testing, `random_state=42` (dashboard ke `utils.py` me bhi yahi split hai, isliye dashboard aur notebooks ka test set same rehta hai).

In [ ]:
X = salary_df[SELECTED_FEATURES].copy()
y = salary_df[TARGET].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 8. Splits aur feature list save karna
Notebook 05 aur 06 inhi files ko load karenge. Feature list ko `models/feature_columns.json` me bhi rakhte hain, kyunki prediction ke time columns ka order wahi hona chahiye.

In [ ]:
X_train.to_csv('../datasets/processed/X_train.csv', index=False)
X_test.to_csv('../datasets/processed/X_test.csv', index=False)
y_train.to_csv('../datasets/processed/y_train.csv', index=False)
y_test.to_csv('../datasets/processed/y_test.csv', index=False)
json.dump(SELECTED_FEATURES, open('../models/feature_columns.json', 'w'), indent=2)
print('Saved X_train, X_test, y_train, y_test and feature_columns.json')

## 9. Summary

- Naye (derived) features **nahi banaye**: original project ke models asli columns par train hue the aur dashboard ke inputs bhi wahi hain, isliye feature set same rakha.
- Salary model **54,459 Placed students** par hai: train 43,567, test 10,892.
- Importance me `cgpa`, `coding_skill_score` aur `internships_count` sabse upar hain; branch, college tier jaise features ka salary par asar lagbhag nahi hai (notebook 02 me bhi yahi dikha).